# Defi EquiAlgo - modele de depart (baseline)

**Hackathon Genie & Informatique 2026 - defi 24 h**

Ce carnet charge les donnees, entraine le modele actuellement en production chez
notre partenaire financier, mesure sa performance, et fait un premier audit d'equite.

Il ne corrige rien. C'est votre travail.

Executez-le en entier une fois (Cell > Run All), lisez la section 5, puis partez de la.

---


## 1. Chargement des donnees

Deux fichiers :

| Fichier | Lignes | Contient |
|---|---|---|
| `data/donnees_demandes.csv` | 10 000 | dossiers historiques **avec** `decision_octroi` |
| `data/candidats_evaluation.csv` | 4 000 | nouveaux dossiers, **sans** etiquette - c'est sur ceux-la que vous serez notes |


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    selection_rate,
    true_positive_rate,
)

demandes = pd.read_csv('data/donnees_demandes.csv')
candidats = pd.read_csv('data/candidats_evaluation.csv')

print(f'{len(demandes)} dossiers historiques, {len(candidats)} a evaluer')
demandes.head()


In [ ]:
demandes.info()
print()
print(demandes.describe(include='all').T[['count', 'unique', 'top', 'mean']])


## 2. Un premier regard sur les regions

Le comite a accorde une bourse a 40 % des candidats. Ce taux est-il le meme partout ?


In [ ]:
# Les cinq regions administratives presentes dans le jeu de donnees.
ELOIGNEES = ['Bas-Saint-Laurent', 'Cote-Nord', 'Gaspesie-Iles-de-la-Madeleine']

def groupe_region(df):
    """Regroupe les cinq regions en deux blocs : grands centres / regions eloignees."""
    return np.where(df['region_administrative'].isin(ELOIGNEES), 'Eloignee', 'Centre')

par_region = demandes.groupby('region_administrative').agg(
    candidats=('decision_octroi', 'size'),
    taux_octroi=('decision_octroi', 'mean'),
    cote_r_moyenne=('cote_r_equivalent', 'mean'),
).sort_values('taux_octroi')
print(par_region.round(3))

demandes['groupe'] = groupe_region(demandes)
print()
print(demandes.groupby('groupe')['decision_octroi'].mean().round(3))


Notez l'ecart de cote R entre les regions, puis l'ecart de taux d'octroi.
Est-ce que le second s'explique entierement par le premier ? Gardez la question ouverte.


## 3. Le modele en production

Une foret aleatoire entrainee sur toutes les colonnes disponibles, y compris
`region_administrative`. C'est le modele que l'institution utilise aujourd'hui.


In [ ]:
CATEGORIELLES = ['programme_etudes', 'region_administrative', 'code_postal_3']

def encoder(df_entrainement, df_cible):
    """Encode les colonnes categorielles et aligne les colonnes des deux tableaux.

    L'alignement (reindex) est important : si une categorie est absente d'un des
    deux jeux, get_dummies produit un nombre de colonnes different et le modele
    echoue - ou pire, apprend sur les mauvaises colonnes.
    """
    X = pd.get_dummies(
        df_entrainement.drop(columns=['id_candidat', 'decision_octroi', 'groupe'], errors='ignore'),
        columns=CATEGORIELLES,
    )
    Xc = pd.get_dummies(
        df_cible.drop(columns=['id_candidat', 'decision_octroi', 'groupe'], errors='ignore'),
        columns=CATEGORIELLES,
    ).reindex(columns=X.columns, fill_value=0)
    return X, Xc

# Decoupage interne pour mesurer honnetement la performance.
train, test = train_test_split(
    demandes, test_size=0.3, random_state=42, stratify=demandes['decision_octroi']
)
X_train, X_test = encoder(train, test)
y_train, y_test = train['decision_octroi'], test['decision_octroi']

modele = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
modele.fit(X_train, y_train)
y_pred = modele.predict(X_test)

print(f'Exactitude globale : {accuracy_score(y_test, y_pred):.2%}')
print()
print(classification_report(y_test, y_pred, digits=3))


Environ 88 % d'exactitude. Du point de vue du tableau de bord trimestriel, le modele va tres bien.


## 4. Audit d'equite

Deux mesures, et elles ne disent pas la meme chose :

- **Parite demographique** - est-ce que les deux groupes recoivent des bourses au meme taux ?
- **Egalite des chances** - parmi les candidats qui meritent la bourse, est-ce que les deux
  groupes ont la meme chance de l'obtenir ?

La deuxieme demande de savoir qui merite la bourse. Regardez bien avec quoi on la calcule ici.


In [ ]:
groupe_test = groupe_region(test)

cadre = MetricFrame(
    metrics={
        'exactitude': accuracy_score,
        'taux_selection': selection_rate,
        'taux_vrais_positifs': true_positive_rate,
    },
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=groupe_test,
)
print(cadre.by_group.round(4))

ecart_parite = demographic_parity_difference(y_test, y_pred, sensitive_features=groupe_test)
print(f'\nEcart de parite demographique : {ecart_parite:.4f}')


Le `taux_vrais_positifs` ci-dessus est calcule par rapport a `decision_octroi`,
c'est-a-dire par rapport aux decisions passees du comite.

Autrement dit : on vient de verifier que le modele reproduit fidelement le comite.
Ce n'est pas la meme chose que verifier qu'il est juste.


## 5. Le piege

Le reflexe est de supprimer `region_administrative` et de declarer le probleme regle.
Mesurons.


In [ ]:
essais = {'modele complet': []}

def entrainer_sans(colonnes_exclues, etiquette):
    X, Xt = encoder(train, test)
    garder = [c for c in X.columns if not c.startswith(tuple(colonnes_exclues))] if colonnes_exclues else list(X.columns)
    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
    m.fit(X[garder], y_train)
    p = m.predict(Xt[garder])
    ecart = demographic_parity_difference(y_test, p, sensitive_features=groupe_test)
    print(f'{etiquette:<42} exactitude {accuracy_score(y_test, p):.4f}   ecart parite {ecart:.4f}')
    return ecart

entrainer_sans((), 'toutes les colonnes')
entrainer_sans(('region_administrative_',), 'sans region_administrative')
entrainer_sans(('region_administrative_', 'code_postal_3_'), 'sans region ni code postal')


L'ecart bouge a peine.

D'autres colonnes portent l'information regionale : la distance domicile-campus, les
heures travaillees, le revenu familial, le code postal. Supprimer l'etiquette ne supprime
pas le signal. C'est l'**effet proxy**.

Mesurez-le vous-meme - une matrice de correlation, ou un modele qui essaie de predire la
region a partir des autres colonnes, vous dira lesquelles sont les plus bavardes.


## 6. Votre mission

Produire, pour les 4 000 candidats de `data/candidats_evaluation.csv`, une decision
d'octroi plus juste que celle du modele actuel.

Trois contraintes :

1. **Budget fixe.** L'institution finance un nombre fixe de bourses. Votre taux d'octroi
   global doit rester entre **36 % et 44 %**. Une soumission hors de cette plage recoit 0
   sur la section technique. Accorder la bourse a tout le monde n'est pas une solution.
2. **Vous n'etes pas notes sur `decision_octroi`.** Le jury compare vos decisions a un
   etalon de reference construit independamment du comite historique, que vous n'avez pas.
   Reproduire parfaitement `decision_octroi` reproduit parfaitement le biais.
3. **Format.** Deux colonnes, 4 000 lignes, valeurs 0 ou 1.

La cellule suivante ecrit une soumission valide a partir du modele biaise. Elle sert de
gabarit. Remplacez `predictions` par les votres.


In [ ]:
X_hist, X_cand = encoder(demandes, candidats)
modele_complet = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
modele_complet.fit(X_hist, demandes['decision_octroi'])
predictions = modele_complet.predict(X_cand)

soumission = pd.DataFrame({
    'id_candidat': candidats['id_candidat'],
    'decision_octroi': predictions.astype(int),
})
soumission.to_csv('predictions.csv', index=False)

taux = soumission['decision_octroi'].mean()
print(f'{len(soumission)} lignes ecrites dans predictions.csv')
print(f'Taux d\'octroi : {taux:.1%}', '- OK' if 0.36 <= taux <= 0.44 else '- HORS BUDGET')
soumission.head()


---

### Pistes pour la suite

- Quelles colonnes predisent le mieux la region ? Ce sont vos proxys.
- Entre parite demographique et egalite des chances, laquelle defendriez-vous devant un
  comite d'ethique, et pourquoi ? Les deux ne peuvent pas etre satisfaites en meme temps
  si les deux groupes n'ont pas exactement le meme profil.
- `fairlearn.postprocessing.ThresholdOptimizer` et `fairlearn.reductions.ExponentiatedGradient`
  sont les deux outils les plus rapides a mettre en place. Le premier ajuste les seuils apres
  coup, le second reentraine sous contrainte.
- Faites varier la contrainte d'equite et tracez le nuage (equite, exactitude) : c'est votre
  front de Pareto, et le jury va vous demander de le defendre.
- Si `decision_octroi` est une etiquette biaisee, est-ce que le meilleur modele est celui qui
  la predit le mieux ?
